In [1]:
%pip -q install ucimlrepo scikit-learn pandas numpy matplotlib seaborn

In [2]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
Pandas: 2.2.3
Scikit-learn: 1.6.1
NumPy: 2.4.6


In [3]:
from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=2)

X = adult.data.features.copy()
y_raw = adult.data.targets.copy()

print("Features shape:", X.shape)
print("Target shape:", y_raw.shape)

display(X.head())
display(y_raw.head())

Features shape: (48842, 14)
Target shape: (48842, 1)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba


,income
0,<=50K
1,<=50K
2,<=50K
3,<=50K
4,<=50K


In [4]:
# Inspect the complete target distribution

print("Target labels:")
print(y_raw.iloc[:, 0].value_counts(dropna=False))

print("\nUnique target values:")
print(y_raw.iloc[:, 0].unique())

print("\nMissing target values:")
print(y_raw.isna().sum())

Target labels:
income
<=50K     24720
<=50K.    12435
>50K       7841
>50K.      3846
Name: count, dtype: int64

Unique target values:
['<=50K' '>50K' '<=50K.' '>50K.']

Missing target values:
income    0
dtype: int64


In [5]:
# Standardise target labels

y_text = (
    y_raw.iloc[:, 0]
    .astype(str)
    .str.strip()
    .str.rstrip(".")
    .str.strip()
)

# Convert to binary classification
label_map = {
    "<=50K": 0,
    ">50K": 1
}

y = y_text.map(label_map)

# Validation
print("Normalised target distribution:")
print(y.value_counts().sort_index())

print("\nTarget mapping:")
print("0 = Income <=50K")
print("1 = Income >50K")

print("\nMissing target values:", y.isna().sum())
print("Total records:", len(y))

assert y.notna().all()
assert set(y.unique()) == {0, 1}
assert len(y) == len(X)

Normalised target distribution:
income
0    37155
1    11687
Name: count, dtype: int64

Target mapping:
0 = Income <=50K
1 = Income >50K

Missing target values: 0
Total records: 48842


In [6]:
# DQ-002: Baseline Data Quality Assessment

import pandas as pd
import numpy as np

# 1. Dataset dimensions
print("1. DATASET DIMENSIONS")
print("Rows:", X.shape[0])
print("Columns:", X.shape[1])

# 2. Missing values
print("\n2. MISSING VALUE ANALYSIS")

missing = pd.DataFrame({
    "Missing_Count": X.isna().sum(),
    "Missing_Percentage": X.isna().mean() * 100
}).sort_values("Missing_Percentage", ascending=False)

display(missing)

# 3. Duplicate records
print("\n3. DUPLICATE ANALYSIS")

print("Exact duplicate feature rows:", X.duplicated().sum())

# 4. Data types
print("\n4. DATA TYPES")
display(X.dtypes.to_frame("Data_Type"))

# 5. Unique categorical values
print("\n5. CATEGORICAL UNIQUENESS")

categorical_cols = X.select_dtypes(
    include=["object", "string", "category"]
).columns

for col in categorical_cols:
    print(f"{col}: {X[col].nunique(dropna=True)} unique values")

# 6. Numerical summary
print("\n6. NUMERICAL SUMMARY")
display(X.describe().T)

# 7. Target distribution
print("\n7. TARGET DISTRIBUTION")
display(y.value_counts(normalize=True).rename("Proportion"))

print("\nBASELINE PROFILING COMPLETED")

1. DATASET DIMENSIONS
Rows: 48842
Columns: 14

2. MISSING VALUE ANALYSIS


,Missing_Count,Missing_Percentage
occupation,966,1.977806
workclass,963,1.971664
native-country,274,0.560993
education,0,0.000000
fnlwgt,0,0.000000
age,0,0.000000
marital-status,0,0.000000
education-num,0,0.000000
race,0,0.000000
relationship,0,0.000000



3. DUPLICATE ANALYSIS
Exact duplicate feature rows: 53

4. DATA TYPES


,Data_Type
age,int64
workclass,object
fnlwgt,int64
education,object
education-num,int64
marital-status,object
occupation,object
relationship,object
race,object
sex,object



5. CATEGORICAL UNIQUENESS
workclass: 9 unique values
education: 16 unique values
marital-status: 7 unique values
occupation: 15 unique values
relationship: 6 unique values
race: 5 unique values
sex: 2 unique values
native-country: 42 unique values

6. NUMERICAL SUMMARY


,count,mean,std,min,25%,50%,75%,max
age,48842.0,38.643585,13.710510,17.0,28.0,37.0,48.0,90.0
fnlwgt,48842.0,189664.134597,105604.025423,12285.0,117550.5,178144.5,237642.0,1490400.0
education-num,48842.0,10.078089,2.570973,1.0,9.0,10.0,12.0,16.0
capital-gain,48842.0,1079.067626,7452.019058,0.0,0.0,0.0,0.0,99999.0
capital-loss,48842.0,87.502314,403.004552,0.0,0.0,0.0,0.0,4356.0
hours-per-week,48842.0,40.422382,12.391444,1.0,40.0,40.0,45.0,99.0



7. TARGET DISTRIBUTION


,Proportion
income,
0,0.760718
1,0.239282



BASELINE PROFILING COMPLETED


In [7]:
# Baseline quality summary

print("MISSING VALUES")
print(X.isna().sum().sort_values(ascending=False))

print("\nMISSING VALUE TOTAL:", X.isna().sum().sum())

print("\nEXACT DUPLICATE ROWS:", X.duplicated().sum())

print("\nDATA TYPES")
print(X.dtypes)

print("\nCLASS DISTRIBUTION")
print(y.value_counts())

MISSING VALUES
occupation        966
workclass         963
native-country    274
education           0
fnlwgt              0
age                 0
marital-status      0
education-num       0
race                0
relationship        0
sex                 0
capital-gain        0
capital-loss        0
hours-per-week      0
dtype: int64

MISSING VALUE TOTAL: 2203

EXACT DUPLICATE ROWS: 53

DATA TYPES
age                int64
workclass         object
fnlwgt             int64
education         object
education-num      int64
marital-status    object
occupation        object
relationship      object
race              object
sex               object
capital-gain       int64
capital-loss       int64
hours-per-week     int64
native-country    object
dtype: object

CLASS DISTRIBUTION
income
0    37155
1    11687
Name: count, dtype: int64


In [8]:
# DQ-003: Feature-level missing value analysis

missing_report = pd.DataFrame({
    "Missing_Count": X.isna().sum(),
    "Missing_Percentage": X.isna().mean() * 100,
    "Data_Type": X.dtypes.astype(str)
})

missing_report = missing_report.sort_values(
    "Missing_Count",
    ascending=False
)

display(missing_report)

print("Total missing cells:",
      missing_report["Missing_Count"].sum())

print("Rows containing at least one missing value:",
      X.isna().any(axis=1).sum())

print("Rows with no missing values:",
      (~X.isna().any(axis=1)).sum())

,Missing_Count,Missing_Percentage,Data_Type
occupation,966,1.977806,object
workclass,963,1.971664,object
native-country,274,0.560993,object
education,0,0.000000,object
fnlwgt,0,0.000000,int64
age,0,0.000000,int64
marital-status,0,0.000000,object
education-num,0,0.000000,int64
race,0,0.000000,object
relationship,0,0.000000,object


Total missing cells: 2203
Rows containing at least one missing value: 1221
Rows with no missing values: 47621


In [9]:
# DQ-004: Duplicate record investigation

duplicate_mask = X.duplicated(keep=False)

duplicate_data = pd.DataFrame({
    "Feature_Duplicate": duplicate_mask,
    "Target": y
})

print("Rows involved in duplicate feature groups:",
      duplicate_mask.sum())

print("\nTarget distribution among duplicate rows:")

display(
    duplicate_data[duplicate_mask]["Target"]
    .value_counts()
)

print("\nFull-record duplicates (features + target):")

full_data = X.copy()
full_data["income_target"] = y

print(full_data.duplicated().sum())

Rows involved in duplicate feature groups: 103

Target distribution among duplicate rows:


,count
Target,
0,86
1,17



Full-record duplicates (features + target):
48


In [10]:
# DQ-005: Identify conflicting labels among duplicate features

duplicate_analysis = X.copy()
duplicate_analysis["target"] = y.values

group_analysis = (
    duplicate_analysis
    .groupby(list(X.columns), dropna=False)
    .agg(
        Record_Count=("target", "size"),
        Unique_Targets=("target", "nunique"),
        Target_Values=("target", lambda x: sorted(x.unique()))
    )
    .reset_index()
)

duplicate_groups = group_analysis[
    group_analysis["Record_Count"] > 1
]

conflicting_groups = duplicate_groups[
    duplicate_groups["Unique_Targets"] > 1
]

print("Repeated feature groups:", len(duplicate_groups))

print("Conflicting target groups:", len(conflicting_groups))

print("\nConflicting records:")
display(conflicting_groups)

print("\nRepeated feature rows:")
print(duplicate_groups["Record_Count"].sum())

print("\nExact duplicate occurrences:")
print(
    duplicate_groups["Record_Count"].sum()
    - len(duplicate_groups)
)

Repeated feature groups: 50
Conflicting target groups: 5

Conflicting records:


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,Record_Count,Unique_Targets,Target_Values
20234,34,Private,169583,Masters,14,Never-married,Exec-managerial,Not-in-family,White,Female,0,0,50,United-States,2,2,"[0, 1]"
26621,39,Private,138192,Bachelors,13,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,2,2,"[0, 1]"
29392,41,Private,242804,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,40,United-States,2,2,"[0, 1]"
34954,46,Private,224559,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,Black,Male,0,0,40,United-States,2,2,"[0, 1]"
34971,46,Private,235646,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,2,2,"[0, 1]"



Repeated feature rows:
103

Exact duplicate occurrences:
53
